# Lab 10: Classifying real supernovae, rare ones included

**ASTR 457, Fall 2026, posted Thu Nov 19, due Wed Dec 2 by noon (fork → PR, as always). Nothing is due over fall break; this lab has two weeks for that reason.**

On Nov 10 we classified simulated Rubin supernovae (ELAsTiCC) as Ia or II with trees and forests, and on Nov 12 we saw what happens when one class is rare, when some values are missing, and when nobody has labeled anything at all. This time the supernovae are real. Your files come from the Zwicky Transient Facility's Bright Transient Survey (BTS; Fremling et al. 2020, Perley et al. 2020), which takes a spectrum of every transient brighter than about 18.5 mag, so every object has a type from a spectrum. You'll build classifiers from light-curve properties, find the rare ones, and referee an AI's attempt at the same job.

Put your notebook in `submissions/<your netid>/lab10.ipynb`, and your predictions in `submissions/<your netid>/lab10_predictions.csv` (Part 5). The data paths below are relative to that folder. The notebook has to run top to bottom from a fresh kernel ("Restart & Run All").

Part 1 is 15%, Part 2 25%, Part 3 20%, Part 4 25%, Part 5 5%, and the AI-use appendix and verification plan 10%. Quote every number with an uncertainty where it has one (the spread over cross-validation folds counts) and a sensible number of digits.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

NETID = 'yournetid'   # <-- set this
LAB = '../../labs/10/data/'                               # from submissions/<your netid>/
train = pd.read_csv(LAB + f'{NETID}_train.csv')           # 5,305 objects with their spectroscopic type
test = pd.read_csv(LAB + f'{NETID}_test.csv')             # 1,500 objects, type withheld (Part 5)

## The data

Each row is one transient. The columns are the BTS catalog's own (see the [BTS sample explorer](https://sites.astro.caltech.edu/ztf/bts/explorer.php) for its definitions):

| column | what it is |
|---|---|
| `peakfilt` | the ZTF filter (`g` or `r`) in which the light curve peaked |
| `peakmag` | the brightest magnitude ZTF measured, in that filter (fainter than the true peak if the peak was missed) |
| `peakabs` | absolute magnitude at that peak, from `peakmag`, the redshift and the Galactic extinction, with a rough k-correction |
| `rise`, `fade` | days from half the peak flux to the peak, and from the peak back down to half, read off the light curve by linear interpolation |
| `duration` | `rise` + `fade`, in days |
| `redshift` | the redshift (the BTS notes it is often approximate) |
| `b` | Galactic latitude, degrees |
| `A_V` | Galactic extinction toward the object, mag |
| `type` | the spectroscopic classification (training file only) |

Throughout, **"Ia"** means any type that starts with `SN Ia`, and **"rare"** means any type that starts with `SLSN` (superluminous supernovae) or `TDE` (tidal disruption events, a star torn apart by a black hole). Everything else is "other".

These are your training data: 5,305 objects with labels. Your test file has 1,500 more objects from the same catalog, with the type removed. I have their types, and Part 5 is graded on them. The BTS catalog is public, so you could look the test objects up; don't. Your predictions get compared with your notebook, so they have to come from it.

## Part 1: Look first (15%)

1. How many Ia, rare and other objects are in your training file? If a classifier said "Ia" for every object, what accuracy would it get? What about "not rare" for every object?
2. Plot `fade` against `rise` and histogram `peakabs`, colored by Ia / rare / other. Histogram `peakmag`. Describe what you see. Which of these would separate the classes on their own?
3. Some values in the file aren't numbers. Find them, say what they mean, and count, for each class, the objects that have at least one. Then say what you'll do with them, and why. (Nov 12: is dropping them censoring or truncation, and which classes would it remove?)

#### Hints:
- `pd.read_csv` will read a column with even one non-number in it as strings. Check `train.dtypes` before you plot anything.
- `pd.to_numeric(s, errors='coerce')` turns anything it can't read into NaN. Know what you threw away before you use it.

In [ ]:
# Part 1: your work here

**ANSWER (class counts and the do-nothing accuracies, what the plots show, and the non-numbers: what they are, how many, what you do with them):**

*your answer goes here*

## Part 2: Ia or not (25%)

A cosmologist wants a sample of type Ia supernovae with as few impostors as possible (Lab 09 measured $\Omega_m$ from just such a sample, contaminated).

1. Split your training file into a training set and a validation set before you do anything else, and say how you split it. Keep the validation set out of everything until 2.4.
2. Fit a decision tree for Ia vs not-Ia. Choose its depth by 5-fold cross-validation on the training set and plot the cross-validated score against depth. Then fit a random forest.
3. Fit each model twice: with every column, and with only the columns that don't need a spectrum (what a survey like Rubin would have from its own light curve, before anyone takes a spectrum). Say which columns those are, and why.
4. On the validation set, for each model, report the Ia purity and completeness at a threshold of 0.5, plot the ROC curves together, and compare with the do-nothing classifier from Part 1.
5. Choose a threshold for the cosmologist's sample. What fraction of the objects you call Ia are impostors (Lab 09's $f$), and what fraction of the real Ia do you lose? Which features matter most to your forest, and does that make physical sense?

#### Hints:
- `sklearn.model_selection.train_test_split(..., stratify=...)`, `cross_val_score`, `sklearn.metrics.precision_score`, `recall_score`, `roc_curve`.
- Precision = purity, recall = completeness (Nov 10).

In [ ]:
# Part 2: your work here

**ANSWER (your split, the depth, purity and completeness with and without the spectroscopic columns, your threshold, f, and the feature importances):**

*your answer goes here*

## Part 3: The rare ones (20%)

Superluminous supernovae and tidal disruption events are about 2% of your file, and they're the objects someone would want a spectrum of first.

1. Train a random forest for rare vs not-rare as it is, and report its accuracy, purity and completeness on your validation set. Why is the accuracy not the number to quote?
2. Do better at finding them. Try at least two of: class weights, undersampling the common classes, oversampling the rare class, a lower threshold (Nov 12). Report the purity and completeness of each on the same validation set, and pick one. Whatever you resample, resample only the training set: say why.

   Your validation set has only about 30 rare objects, so one more or one fewer found moves the completeness by 0.03. Give your numbers uncertainties (cross-validation helps), and don't pick a method on a difference smaller than that.
3. Now forget the labels. Run an isolation forest on the light-curve columns of your whole training file and look at the 50 most anomalous objects. How many are rare? What are the others, and why did the isolation forest flag them? (Nov 12: the most anomalous object in the M31 field was a bad measurement.)

In [ ]:
# Part 3: your work here

**ANSWER (the as-is rare classifier, your two or more fixes and your pick, and what the isolation forest found):**

*your answer goes here*

## Part 4: Referee the AI (25%)

`labs/10/ai_solution.ipynb` is an AI's analysis of the same kind of file. Copy it into your folder, point it at your own data (set its `NETID`, and change its `data/` paths to `../../labs/10/data/`), and run it. It is well written and it runs. Find what is wrong with it.

For each problem you find: say what it is, **demonstrate it with numbers on your own file** (a comparison, not an opinion), and say what the AI should have done. Full credit needs at least three, demonstrated. If you think a claim is wrong but your file can't show it, say so and show what you tried.

In [ ]:
# Part 4: your work here

**ANSWER (each problem: what it is, your demonstration on your file, the fix):**

*your answer goes here*

## Part 5: Your predictions (5%)

Apply your best Ia classifier and your best rare classifier to your 1,500 test objects. Write `lab10_predictions.csv` beside your notebook, with one row per test object and the columns `obj`, `p_ia`, `is_ia`, `p_rare`, `is_rare` (the two `p_` columns are your classifiers' probabilities, and the two `is_` columns are 0 or 1 at the thresholds you chose in Parts 2 and 3). Predict how your Ia purity and completeness and your rare purity and completeness will come out on the test set, with uncertainties.

You'll be graded on whether your predictions are as good as you said they'd be, not on beating anyone.

In [ ]:
# Part 5: your work here (write lab10_predictions.csv)

**ANSWER (which models, the thresholds, and your predicted test purity and completeness for Ia and for rare):**

*your answer goes here*

## Part 6: AI-use appendix and verification plan (10%)

1. **AI use**: which tools, for what, what they got wrong, and how you caught it. "I didn't use any" is fine if true.
2. **Reflection**: what would have gone wrong in your answer if you had not checked the AI's output? Name the specific thing.
3. **Verification plan**: the checks you ran before trusting your Part 2 and 3 numbers, and what failure would have looked like for each. One to consider: shuffle the labels in your training set and rerun. What should the purity and completeness be, and are they?

**AI-USE APPENDIX:**

*your answer goes here*

**REFLECTION:**

*your answer goes here*

**VERIFICATION PLAN:**

*your plan goes here*